# CR-2 G0・G1：土台の道（Claude API・CPU）

凍結した計画の順に、**G1 の前の受け入れ 2 → G0 → G1 → G1b → 段3 の材料**を行います。

- **G1 の前の受け入れ 2**（セル2）：受け入れ 1（試験と人工データ）の結果がコードの固定と一致し、そのあとコードが変わっていないことを確かめます。データを読み直して、ファイルの証跡・段2 の候補の一覧と照合します。どれかが違えば 1 件も送りません。
- **G0**（セル3、上限 $0.50）：無害な文だけで、最大 1,024 トークンと count_tokens が通ることを確かめます。問題は送りません。
- **G1**（セル4、上限 $3）：候補 120 問の土台の道を 1 回ずつ作ります（温度 0）。条件（答えが読める・end_turn・40〜300 単位）を満たした問題から、**決まった順の先頭から** G1b で N（60〜90）を決めます。途中で切れても、もう一度「すべてのセルを実行」で続きから回ります。
- G1b が GO なら、段3 の材料（探す段の 1 回目のリクエストの一覧のハッシュなど）を Drive に書きます。Claude がそれをリポジトリに固定してから G2s です。
- 問題文・応答は表示しません。土台の道の文は Drive の非公開ファイル（`*_PRIVATE.json`）にだけ保存します。

**使い方**：CPU のランタイム。シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。「すべてのセルを実行」で 10〜30 分。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, ancestors, (pinned SDK and key, never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "311e3008211ebaca0735abccf3501e6fcb8da3df"
ANCESTORS = ["f129850e075c0228c35749f690ec3f9525554bc7", "311e3008211ebaca0735abccf3501e6fcb8da3df", "a9fd5344084cdec8c402f1768cdf993056a7b86f"]
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in ["7949d74"] + ANCESTORS:
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from pork import cr2 as C
from pork import cr2_search as S
from pork import cr2_run as RN
from pork import cr2_g2 as G
from pork import cr2_g3 as G3
from pork import cr2_stage3b as T3
from pork import cr1_run as R1
from pork import alt_prelook as P
import numpy
RN.check_prompt()
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr2")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))

def git_blob(path):
    return subprocess.check_output(["git", "hash-object", path], text=True).strip()

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    R1._atomic_write(path, obj)
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_chase_verified():
    """Plan 1. (H4, I3): pinned revision, canonical hash from the Hugging Face API vs the downloaded bytes."""
    from huggingface_hub import HfApi, hf_hub_download
    info = HfApi().dataset_info(C.DATASET_ID, revision=C.DATASET_REVISION, files_metadata=True, token=False)
    sib = next((s for s in info.siblings if s.rfilename == C.DATASET_FILE), None)
    meta = None
    if sib is not None:
        lfs = sib.lfs
        if lfs is not None and not isinstance(lfs, dict):
            lfs = {"sha256": getattr(lfs, "sha256", None), "size": getattr(lfs, "size", None)}
        meta = {"blob_id": sib.blob_id, "lfs": lfs}
    p = hf_hub_download(C.DATASET_ID, C.DATASET_FILE, repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    data = Path(p).read_bytes()
    prov = C.check_provenance(data, meta, info.sha)
    rp = hf_hub_download(C.DATASET_ID, "README.md", repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    readme = Path(rp).read_text(encoding="utf-8")
    rows = C.parse_rows(data)
    del data
    return rows, prov, {"readme_sha256": C.sha256_text(readme), "license": C.readme_license(readme)}

def rebuild_stage2(file_sha=None, cand_sha=None, excl_sha=None):
    rows, prov, readme = load_chase_verified()
    if file_sha is not None:
        assert prov["file_sha256"] == file_sha, "データのファイルが段2 の固定と違います。止めます（1 件も送りません）。"
    pre = C.reproduce_prelook3(rows, P.load_math())
    cand = C.build_candidates(rows, pre)
    if cand_sha is not None:
        assert cand["candidates_sha256"] == cand_sha, "候補の一覧が段2 の固定と違います。止めます。"
    if excl_sha is not None:
        assert cand["excluded_sha256"] == excl_sha, "除外した行が段2 の固定と違います。止めます。"
    by_id = {int(r["ID"]): r for r in rows}
    cands = [{**c, "question": C.problem_text(by_id[c["id"]]), "gold": str(C.gold(by_id[c["id"]]))}
             for c in cand["candidates"]]                                     # PRIVATE, memory only
    del rows, by_id
    return cand, cands, prov, readme, pre

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RN.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RN.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key)                       # reads, count_tokens: default retries
create_client = anthropic.Anthropic(api_key=_key, max_retries=0)  # sends: never retried by the SDK
del _key
TRANSIENT = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)
print("anthropic", anthropic.__version__)

print("study:", C.STUDY, "| exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-2 G0・G1 土台の道（API・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-2 G0 G1', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — G1 の前の受け入れ 2.  Nothing is sent before this passes.
CANDIDATES_SHA = "c4bffbfa9632ed51a1222fae0bc65b256b3db0ae7e1b0d12e8bf70ac649f2339"
EXCLUDED_SHA = "5323fa0f180009c9a8c85b8885a1800e79f4a4db026ac21edee3162e017edab5"
FILE_SHA = "731f2db9a590ac6c49ab7529d08dee0fcb09cc832a6462248e6939919319222c"
PREG1_HEAD = "a9fd5344084cdec8c402f1768cdf993056a7b86f"
PREG1_SHA = "920544a3476205c2285b35739ef12724e9779c773b7933bcec3118fcea20c0c5"
pf = D / "PREG1/CR2_preg1_result.json"
assert hashlib.sha256(pf.read_bytes()).hexdigest() == PREG1_SHA, "受け入れ 1 の結果がコードの固定と違います。止めます。"
pre1 = json.loads(pf.read_text())
assert pre1["pass"] and pre1["tests_passed"] and pre1["code_head"] == PREG1_HEAD, "受け入れ 1 が合格していません。止めます。"
diff = subprocess.run(["git", "diff", "--quiet", PREG1_HEAD, head, "--", "pork", "scripts", "tests"])
assert diff.returncode == 0, "受け入れ 1 のあとでコードが変わっています。止めます。"
cand, cands, prov, readme, pre = rebuild_stage2(FILE_SHA, CANDIDATES_SHA, EXCLUDED_SHA)
L2 = json.loads((REPO / "notes/cr2_lock_stage2.json").read_text())                     # the committed 段2 lock
for k, got in (("file_sha256", prov["file_sha256"]), ("candidates_sha256", cand["candidates_sha256"]),
               ("prelook3_excluded_ids_sha256", cand["excluded_sha256"]), ("dedup_dropped", cand["dedup_dropped"]),
               ("prelook3_excluded", cand["excluded"]), ("rows", cand["rows"]), ("prelook3_all96_sha256", pre["all96_sha256"])):
    assert L2[k] == got, "段2 の固定（" + k + "）と違います。止めます。"
assert L2["provenance"] == prov and L2["GL_RECORD"]["readme_sha256"] == readme["readme_sha256"], "ファイルの証跡か README が段2 の固定と違います。止めます。"
print("受け入れ 2: OK | 受け入れ 1 の試験:", pre1["tests"], "| ファイル・候補の照合 OK | 候補", len(cands))

In [ ]:
# Cell 3 — G0 (cap $0.50): harmless text only; max_tokens 1,024 and count_tokens with a prefill.
G0_PATH = D / "G0/CR2_G0.json"
if G0_PATH.exists():
    g0 = json.loads(G0_PATH.read_text())
    assert g0["code_head"] == head and g0["pass"], "G0 の記録が別の実行のものか、不合格です。Claude に教えてね。"
    print("G0 はすみ: $%.4f" % g0["spent_usd"])
else:
    HARMLESS = "What is 1 + 2?"
    m = create_client.messages.create(**C.baseline_params(HARMLESS))
    n_tok = client.messages.count_tokens(model=C.MODEL, messages=C.resample_params(HARMLESS, "First,")["messages"]).input_tokens
    spent = RN.cost_usd(m.usage.input_tokens, m.usage.output_tokens)
    g0 = {"code_head": head, "model": m.model, "stop_reason": m.stop_reason, "in": m.usage.input_tokens,
          "out": m.usage.output_tokens, "count_tokens_with_prefill": n_tok, "prompt_sha256": C.prompt_sha256(),
          "revision": C.DATASET_REVISION, "file_sha256": FILE_SHA, "max_tokens": C.MAX_TOKENS, "spent_usd": spent,
          "candidates_sha256": cand["candidates_sha256"], "excluded_sha256": cand["excluded_sha256"]}
    g0["pass"] = (m.model == C.MODEL and n_tok > 0 and spent <= RN.G0_CAP_USD)
    write_json(G0_PATH, g0)
    assert g0["pass"], "G0 が不合格です。止めます。"
    print("G0: OK | モデル", m.model, "| count_tokens", n_tok, "| $%.4f" % spent)

In [ ]:
# Cell 4 — G1, G1b, 段3 material.  Numbers / ids / hashes only.
P_ = RN.paths()
CK, NUM = Path("/content/drive/MyDrive") / P_["checkpoint"], Path("/content/drive/MyDrive") / P_["numbers"]
CK.parent.mkdir(parents=True, exist_ok=True)
R1.drive_smoke(CK.parent)
HDR = RN.header(head, cand["candidates_sha256"], CANDIDATES_SHA, anthropic.__version__, C.DATASET_REVISION, prov["file_sha256"], FILE_SHA)
ck = RN.load_files(CK, NUM, HDR, cands)
import threading
_lk = threading.Lock()
def save():
    with _lk:
        RN.save_files(CK, NUM, ck)
n = RN.run_g1(cands, lambda p: create_client.messages.create(**p), ck, save, retry_on=TRANSIENT)
save()
g1 = RN.summarize_g1(ck, cands, g0["spent_usd"])
write_json(D / "G1/CR2_G1_summary.json", g1)
print("G1: 呼び出し", n, "| 条件を満たした", g1["N_after_G1"], "/ 120 | 正答率 %.2f" % g1["accuracy_all"], "| 費用 $%.3f" % ck["spent_usd"])
for g_, d in g1["by_depth_group"].items():
    print(" 深さ", g_, "| 通過", d["passed"], "/", d["called"], "| 正答率", d["accuracy_all"], "| 読めない", d["unreadable"],
          "| 1,024 で切れた", d["max_tokens"], "| 外れた理由", d["reasons"])
print("判定:", g1["verdict"], g1.get("why", ""))
if g1["verdict"] == "GO":
    print("G1b: N =", g1["N_final"], g1["G1b"]["N_by_depth_group"], "| 見込み $%.2f" % g1["G1b"]["estimate"]["usd"],
          "/ 使える額 $%.2f" % g1["G1b"]["allowance_usd"], "| 種", {k: g1["seeds_final"][k] for k in ("distinct_seeds", "N_eff")})
    questions = {C.cid(c["id"]): c["question"] for c in cands}
    items = RN.manifest_items(g1, ck, questions)
    rows, r0sha = RN.search_round0(items)
    write_json(D / "G2S/CR2_G2S_round0_manifest.json", {"schema": "cr2-g2s-round0-manifest-v1", "manifest_sha256": r0sha, "rows": rows})
    lock3 = RN.lock_pre_g2(g1, r0sha, len(rows), numpy.__version__, head, tests_passed=True, excluded_sha256=EXCLUDED_SHA,
                           file_sha256=FILE_SHA, search_code_blob=git_blob("pork/cr2_search.py"), cr2_code_blob=git_blob("pork/cr2.py"))
    write_json(D / "STAGE3/CR2_stage3_lock.json", lock3)
    print("段3 の材料: 探す段の 1 回目", len(rows), "件 | round0_sha256", r0sha, "| final_items_sha256", g1["final_items_sha256"])
    del items, questions
del cands
msg = '終わりました：CR-2 の G1 と G1b（' + g1["verdict"] + '）を Drive に書きました'
notify(msg, tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()